In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

In [0]:
%sql
CREATE TABLE IF NOT EXISTS audit_validation
(
table_name STRING,
validation_name STRING,
source_count BIGINT,
target_count BIGINT,
status STRING,
remarks STRING,
validation_time TIMESTAMP
)
USING DELTA;

In [0]:
source_count = spark.table("bronze_customers").count()

target_count = spark.table("silver_customers").count()

status = "PASS" if source_count == target_count else "FAIL"

remarks = "Record Counts Matched" if status == "PASS" else "Record Count Mismatch"

audit_df = spark.createDataFrame(
[
(
"Customers",
"Record Count Validation",
source_count,
target_count,
status,
remarks
)
],
[
"table_name",
"validation_name",
"source_count",
"target_count",
"status",
"remarks"
]
).withColumn("validation_time", current_timestamp())

audit_df.write.mode("append").saveAsTable("audit_validation")

display(audit_df)

In [0]:
null_count = spark.table("silver_customers") \
.filter(col("customer_id").isNull()) \
.count()

status = "PASS" if null_count == 0 else "FAIL"

remarks = f"Null Records : {null_count}"

audit_df = spark.createDataFrame(
[
(
"Customers",
"Null Validation",
0,
null_count,
status,
remarks
)
],
[
"table_name",
"validation_name",
"source_count",
"target_count",
"status",
"remarks"
]
).withColumn("validation_time", current_timestamp())

audit_df.write.mode("append").saveAsTable("audit_validation")

display(audit_df)

In [0]:
duplicate_count = spark.table("silver_customers") \
.groupBy("customer_id") \
.count() \
.filter(col("count") > 1) \
.count()

status = "PASS" if duplicate_count == 0 else "FAIL"

remarks = f"Duplicate Records : {duplicate_count}"

audit_df = spark.createDataFrame(
[
(
"Customers",
"Duplicate Validation",
0,
duplicate_count,
status,
remarks
)
],
[
"table_name",
"validation_name",
"source_count",
"target_count",
"status",
"remarks"
]
).withColumn("validation_time", current_timestamp())

audit_df.write.mode("append").saveAsTable("audit_validation")

display(audit_df)

In [0]:
source_columns = set(spark.table("bronze_customers").columns)

target_columns = set(spark.table("silver_customers").columns)

# Ignore audit columns
ignore_columns = {"data_source", "load_timestamp"}

target_columns = target_columns - ignore_columns

missing_columns = source_columns - target_columns
extra_columns = target_columns - source_columns

status = "PASS" if len(missing_columns) == 0 and len(extra_columns) == 0 else "FAIL"

remarks = f"Missing: {list(missing_columns)}, Extra: {list(extra_columns)}"

audit_df = spark.createDataFrame(
[
(
"Customers",
"Schema Validation",
len(source_columns),
len(target_columns),
status,
remarks
)
],
[
"table_name",
"validation_name",
"source_count",
"target_count",
"status",
"remarks"
]
).withColumn("validation_time", current_timestamp())

audit_df.write.mode("append").saveAsTable("audit_validation")

display(audit_df)

In [0]:
source_df = spark.table("bronze_customers").select("customer_id")

target_df = spark.table("silver_customers").select("customer_id")

missing_records = source_df.subtract(target_df)

missing_count = missing_records.count()

status = "PASS" if missing_count == 0 else "FAIL"

remarks = f"Missing Records : {missing_count}"

audit_df = spark.createDataFrame(
[
(
"Customers",
"Source Target Validation",
source_df.count(),
target_df.count(),
status,
remarks
)
],
[
"table_name",
"validation_name",
"source_count",
"target_count",
"status",
"remarks"
]
).withColumn("validation_time", current_timestamp())

audit_df.write.mode("append").saveAsTable("audit_validation")

display(missing_records)

display(audit_df)

In [0]:
invalid_records = spark.table("silver_orders") \
.filter(col("quantity") <= 0)

invalid_count = invalid_records.count()

status = "PASS" if invalid_count == 0 else "FAIL"

remarks = f"Invalid Quantity Records : {invalid_count}"

audit_df = spark.createDataFrame(
[
(
"Orders",
"Business Rule Validation",
0,
invalid_count,
status,
remarks
)
],
[
"table_name",
"validation_name",
"source_count",
"target_count",
"status",
"remarks"
]
).withColumn("validation_time", current_timestamp())

audit_df.write.mode("append").saveAsTable("audit_validation")

display(invalid_records)

display(audit_df)

In [0]:
%sql
SELECT *
FROM audit_validation
ORDER BY validation_time DESC;

In [0]:
print("Bronze Count :", spark.table("bronze_customers").count())

print("Silver Count :", spark.table("silver_customers").count())

In [0]:
from pyspark.sql.functions import col

spark.table("silver_customers") \
.groupBy("customer_id") \
.count() \
.filter(col("count") > 1) \
.display()

In [0]:
silver = spark.table("silver_customers").select("customer_id")

bronze = spark.table("bronze_customers").select("customer_id")

silver.subtract(bronze).display()

In [0]:
bronze.subtract(silver).display()

In [0]:
spark.table("bronze_customers").count()

spark.table("silver_customers").count()

spark.table("silver_customers").groupBy("customer_id").count().filter("count > 1").display()

spark.table("silver_customers").select("customer_id").subtract(
    spark.table("bronze_customers").select("customer_id")
).display()

In [0]:
%sql
DELETE FROM silver_customers
WHERE customer_id = 2001;